# HerWILL Safe Social Media Datathon 2026 — Strong Kaggle Ensemble

This notebook is designed for the supplied `train.csv` / `test.csv`.

### Main model
- **BanglaBERT**: `csebuetnlp/banglabert`
- **Twitter XLM-R**: `cardiffnlp/twitter-xlm-roberta-base`
- Stratified K-fold probability ensemble
- Class-weighted cross entropy
- Early stopping / best-checkpoint selection
- Original Bangla, English, Banglish, emoji and punctuation preserved

### Auxiliary model
- Word + character TF-IDF
- Linear SVM
- Useful for Banglish spelling, slang, obfuscation and short toxic patterns

### Special handling
- Exact normalized-text lookup from training data when the label is unambiguous
- Conflicting duplicate texts are NOT blindly copied
- OOF macro-F1 is used to choose transformer/TF-IDF blend
- Final predictions are checked before writing `submission.csv`

> **Important:** the hidden competition labels are unavailable locally, so the final leaderboard score cannot be known before submission. The notebook reports honest OOF/validation macro-F1 instead.


In [ ]:
# =========================
# 1. CONFIGURATION
# =========================
SEED = 42

# Set these paths to your Kaggle dataset paths.
TRAIN_PATH = "/kaggle/input/YOUR_DATASET/train.csv"
TEST_PATH  = "/kaggle/input/YOUR_DATASET/test.csv"

# If your files are in the working directory, use:
# TRAIN_PATH = "/kaggle/working/train.csv"
# TEST_PATH  = "/kaggle/working/test.csv"

N_FOLDS = 3
MAX_LEN = 192
EPOCHS = 3
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

RUN_BANGLABERT = True
RUN_XLMR = True
RUN_TFIDF = True

# If GPU memory is limited, lower MAX_LEN or TRAIN_BATCH.
# Recommended Kaggle GPU: T4 / P100 / L4 / A100.


In [ ]:
# =========================
# 2. INSTALL / IMPORTS
# =========================
!pip -q install -U transformers datasets accelerate sentencepiece scikit-learn iterative-stratification

import os, re, gc, random, warnings, unicodedata, html, math
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.special import softmax

import torch
from torch import nn
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding, set_seed
)

warnings.filterwarnings("ignore")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 3. LOAD AND AUDIT DATA
# =========================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print("\nTrain columns:", train.columns.tolist())
print("Test columns :", test.columns.tolist())

print("\nMissing values:")
print(train.isna().sum())
print(test.isna().sum())

# Expected schema: train = id, text, y ; test = id, text
# Adapt automatically if the text column has another common name.
def detect_text_col(df):
    for c in ["text", "Text", "comment", "Comment", "content", "Content", "post", "Post"]:
        if c in df.columns:
            return c
    candidates = [c for c in df.columns if df[c].dtype == "object"]
    if candidates:
        return max(candidates, key=lambda c: df[c].fillna("").astype(str).str.len().mean())
    raise ValueError("Could not identify text column.")

TEXT_COL = detect_text_col(train)
if "y" not in train.columns:
    raise ValueError("Training file must contain a 'y' label column.")

print("Using text column:", TEXT_COL)
print("\nLabel counts:")
print(train["y"].value_counts().sort_index())
print("\nLabel proportions:")
print(train["y"].value_counts(normalize=True).sort_index())

print("\nDuplicate raw texts:", train[TEXT_COL].duplicated().sum())


In [ ]:
# =========================
# 4. CAREFUL TEXT NORMALIZATION
# =========================
# Do NOT translate or aggressively remove punctuation/emojis.
# We only repair Unicode/HTML artifacts and whitespace.

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s)

    # Common mojibake fragments can occur in scraped social text.
    # We only attempt UTF-8 repair when it is safe.
    try:
        if any(x in s for x in ["Ã", "Â", "â€", "ðŸ", "à²"]):
            repaired = s.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
            if repaired:
                s = repaired
    except Exception:
        pass

    s = re.sub(r"\s+", " ", s).strip()
    return s

train["text_clean"] = train[TEXT_COL].fillna("").map(normalize_text)
test["text_clean"] = test[TEXT_COL].fillna("").map(normalize_text)

print("Empty train texts:", (train.text_clean == "").sum())
print("Empty test texts :", (test.text_clean == "").sum())

# Inspect very long records.
print("\nLength percentiles:")
print(train.text_clean.str.len().quantile([.5,.9,.95,.99,.999]))


In [ ]:
# =========================
# 5. DUPLICATE / LEAKAGE AUDIT
# =========================
label_sets = train.groupby("text_clean")["y"].agg(lambda x: set(x))
conflicting = label_sets[label_sets.map(len) > 1]

test_to_train = defaultdict(list)
for i, t in enumerate(train.text_clean):
    test_to_train[t].append(i)

exact_test_matches = 0
unique_test_matches = 0

for t in test.text_clean:
    if t in test_to_train:
        exact_test_matches += 1
        labs = set(train.iloc[test_to_train[t]]["y"])
        if len(labs) == 1:
            unique_test_matches += 1

print("Conflicting duplicate-text groups:", len(conflicting))
print("Test texts occurring in train:", exact_test_matches)
print("Test texts with unambiguous train label:", unique_test_matches)


In [ ]:
# =========================
# 6. CLASS WEIGHTS
# =========================
labels = sorted(train["y"].unique())
counts = train["y"].value_counts().sort_index()

# Balanced CE weights, normalized around 1.
class_weights = len(train) / (len(labels) * counts.values.astype(float))
class_weights = class_weights / class_weights.mean()

print("Classes:", labels)
print("Counts:", counts.to_dict())
print("Weights:", dict(zip(labels, class_weights)))


In [ ]:
# =========================
# 7. CUSTOM WEIGHTED TRAINER
# =========================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = (
            torch.tensor(class_weights, dtype=torch.float)
            if class_weights is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


In [ ]:
# =========================
# 8. TRANSFORMER TRAINING FUNCTION
# =========================
def train_transformer_folds(model_name, prefix):
    print("\n" + "="*80)
    print("MODEL:", model_name)
    print("="*80)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    oof = np.zeros((len(train), len(labels)), dtype=np.float32)
    test_probs = np.zeros((len(test), len(labels)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train.text_clean, train.y), 1):
        print(f"\n===== {prefix} FOLD {fold}/{N_FOLDS} =====")

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        tr_df = train.iloc[tr_idx][["text_clean", "y"]].rename(columns={"text_clean":"text", "y":"labels"})
        va_df = train.iloc[va_idx][["text_clean", "y"]].rename(columns={"text_clean":"text", "y":"labels"})
        te_df = test[["text_clean"]].rename(columns={"text_clean":"text"})

        tr_ds = Dataset.from_pandas(tr_df, preserve_index=False)
        va_ds = Dataset.from_pandas(va_df, preserve_index=False)
        te_ds = Dataset.from_pandas(te_df, preserve_index=False)

        def tok(batch):
            return tokenizer(
                batch["text"],
                truncation=True,
                max_length=MAX_LEN
            )

        tr_ds = tr_ds.map(tok, batched=True, remove_columns=["text"])
        va_ds = va_ds.map(tok, batched=True, remove_columns=["text"])
        te_ds = te_ds.map(tok, batched=True, remove_columns=["text"])

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(labels),
            ignore_mismatched_sizes=True
        )

        outdir = f"/kaggle/working/{prefix}_fold{fold}"

        args = TrainingArguments(
            output_dir=outdir,
            learning_rate=LR,
            per_device_train_batch_size=TRAIN_BATCH,
            per_device_eval_batch_size=EVAL_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=EPOCHS,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            logging_steps=100,
            fp16=torch.cuda.is_available(),
            report_to="none",
            save_total_limit=1,
            seed=SEED + fold
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=tr_ds,
            eval_dataset=va_ds,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics,
            class_weights=class_weights
        )
        trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=1))

        trainer.train()

        val_logits = trainer.predict(va_ds).predictions
        te_logits = trainer.predict(te_ds).predictions

        val_prob = softmax(val_logits, axis=1)
        te_prob = softmax(te_logits, axis=1)

        oof[va_idx] = val_prob
        test_probs += te_prob / N_FOLDS

        fold_pred = val_prob.argmax(1)
        print("Fold macro-F1:", f1_score(train.iloc[va_idx].y, fold_pred, average="macro"))

        del trainer, model, tokenizer, tr_ds, va_ds, te_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_pred = oof.argmax(1)
    print("\nOOF", prefix, "macro-F1:",
          f1_score(train.y, oof_pred, average="macro"))
    print(classification_report(train.y, oof_pred, digits=4))

    return oof, test_probs


In [ ]:
# =========================
# 9. RUN BANGLABERT
# =========================
if RUN_BANGLABERT:
    bangla_oof, bangla_test = train_transformer_folds(
        "csebuetnlp/banglabert",
        "banglabert"
    )
else:
    bangla_oof = np.zeros((len(train), 3), dtype=np.float32)
    bangla_test = np.zeros((len(test), 3), dtype=np.float32)


In [ ]:
# =========================
# 10. RUN SOCIAL-MEDIA XLM-R
# =========================
if RUN_XLMR:
    xlmr_oof, xlmr_test = train_transformer_folds(
        "cardiffnlp/twitter-xlm-roberta-base",
        "twitter_xlmr"
    )
else:
    xlmr_oof = np.zeros((len(train), 3), dtype=np.float32)
    xlmr_test = np.zeros((len(test), 3), dtype=np.float32)


In [ ]:
# =========================
# 11. WORD + CHARACTER TF-IDF BASELINE
# =========================
def train_tfidf():
    print("Training TF-IDF ensemble...")

    # Character model is especially useful for Banglish / spelling variants.
    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=250000,
        sublinear_tf=True
    )

    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=150000,
        sublinear_tf=True
    )

    Xc = char_vec.fit_transform(train.text_clean)
    Tc = char_vec.transform(test.text_clean)

    Xw = word_vec.fit_transform(train.text_clean)
    Tw = word_vec.transform(test.text_clean)

    X = __import__("scipy").sparse.hstack([Xc, Xw]).tocsr()
    T = __import__("scipy").sparse.hstack([Tc, Tw]).tocsr()

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(train), len(labels)), dtype=np.float32)
    test_p = np.zeros((len(test), len(labels)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, train.y), 1):
        clf = LinearSVC(
            C=2.0,
            class_weight="balanced",
            random_state=SEED + fold
        )
        clf.fit(X[tr_idx], train.y.iloc[tr_idx])

        va_dec = clf.decision_function(X[va_idx])
        te_dec = clf.decision_function(T)

        # Softmax makes SVM decision scores blendable.
        oof[va_idx] = softmax(va_dec, axis=1)
        test_p += softmax(te_dec, axis=1) / N_FOLDS

    print("TF-IDF OOF macro-F1:",
          f1_score(train.y, oof.argmax(1), average="macro"))

    return oof, test_p

if RUN_TFIDF:
    tfidf_oof, tfidf_test = train_tfidf()
else:
    tfidf_oof = np.zeros((len(train), 3), dtype=np.float32)
    tfidf_test = np.zeros((len(test), 3), dtype=np.float32)


In [ ]:
# =========================
# 12. OOF BLEND SEARCH
# =========================
def normalize_probs(p):
    p = np.asarray(p)
    return p / np.clip(p.sum(axis=1, keepdims=True), 1e-9, None)

bangla_oof = normalize_probs(bangla_oof)
xlmr_oof = normalize_probs(xlmr_oof)
tfidf_oof = normalize_probs(tfidf_oof)

best = (-1, None)

# Grid search is deliberately simple and reproducible.
for wb in np.arange(0.0, 1.01, 0.1):
    for wx in np.arange(0.0, 1.01 - wb, 0.1):
        wt = 1.0 - wb - wx
        blend = wb * bangla_oof + wx * xlmr_oof + wt * tfidf_oof
        score = f1_score(train.y, blend.argmax(1), average="macro")
        if score > best[0]:
            best = (score, (wb, wx, wt))

print("Best OOF blend macro-F1:", best[0])
print("Weights (BanglaBERT, XLM-R, TF-IDF):", best[1])

wb, wx, wt = best[1]

final_oof = wb * bangla_oof + wx * xlmr_oof + wt * tfidf_oof
print("\nBlended OOF classification report:")
print(classification_report(train.y, final_oof.argmax(1), digits=4))


In [ ]:
# =========================
# 13. EXACT-MATCH RETRIEVAL
# =========================
# Only use an exact normalized-text label when ALL matching train examples
# agree. Conflicting duplicate groups are left to the ensemble.

lookup = {}
conflict_count = 0

for text, grp in train.groupby("text_clean"):
    vals = grp.y.unique().tolist()
    if len(vals) == 1:
        lookup[text] = int(vals[0])
    else:
        conflict_count += 1

final_test_prob = (
    wb * normalize_probs(bangla_test) +
    wx * normalize_probs(xlmr_test) +
    wt * normalize_probs(tfidf_test)
)

final_pred = final_test_prob.argmax(1)

retrieved = 0
for i, text in enumerate(test.text_clean):
    if text in lookup:
        final_pred[i] = lookup[text]
        retrieved += 1

print("Unambiguous exact-match replacements:", retrieved)
print("Conflicting groups ignored:", conflict_count)
print("Final prediction distribution:", Counter(final_pred))


In [ ]:
# =========================
# 14. SUBMISSION FILE
# =========================
submission = pd.DataFrame({
    "id": test["id"],
    "y": final_pred.astype(int)
})

assert len(submission) == len(test)
assert submission["y"].notna().all()
assert set(submission["y"].unique()).issubset(set(labels))

SUB_PATH = "/kaggle/working/submission.csv"
submission.to_csv(SUB_PATH, index=False)

print("Saved:", SUB_PATH)
print(submission.head(10))
print("\nSubmission shape:", submission.shape)
print("Missing y:", submission["y"].isna().sum())
print("Labels:", sorted(submission.y.unique()))

# Optional: display class counts
print("\nSubmission label counts:")
print(submission.y.value_counts().sort_index())


## Recommended Kaggle run

For the strongest practical attempt:

- Use a Kaggle GPU.
- Keep `N_FOLDS = 3`.
- Start with `EPOCHS = 3`.
- If GPU time allows, try `EPOCHS = 4` and compare OOF macro-F1.
- Keep the original text; do not translate the whole dataset.
- If memory errors occur, reduce `MAX_LEN` from 192 to 160 and/or `TRAIN_BATCH` from 8 to 4.
- The final file is `/kaggle/working/submission.csv`.

The 70+ target is plausible but **not guaranteed**. Comparable Bangla shared-task work has reported BanglaBERT macro-F1 around 0.71 on a three-class violence dataset, and a later study reported stronger results from transformer ensembles for Bangla hate-speech subtasks. Those are reference points, not predictions for this competition. citeturn0search18turn0search1
